# M5 large (item × store, daily $, ≥ 0.5 units/day) — for Kaggle

Experiments for showing whether the BUS→BS ensembles find **subtle** local patterns:
0. real data: every measure × search × pruning, plus a **500-run** permutation DFD (Bart §4.6);
1. **planted subgroups** of known size and effect (level / trend / timing, 1–10% of normal sales, with outliers);
2. **interaction-only** planted subgroups that greedy beam search can miss;
3. **anytime / equal-budget** curves: how many planted subgroups each method finds within B evaluations or t seconds.

Results are written to `results_v3/` and compared across datasets in `compare_subtle.ipynb`.

**Kaggle:** add the competition data `m5-forecasting-accuracy` to the notebook (it is read from
`/kaggle/input/m5-forecasting-accuracy`); results go to `/kaggle/working/results_v3`. Locally it reads `M5/`.

In [ ]:
import os
import time
import heapq
import itertools
from collections import Counter

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 110)
N_JOBS = os.cpu_count()
print("CPU cores:", N_JOBS)

In [ ]:
DATASET = "m5_large"
QUICK = False          # True = tiny grids, only to test the notebook end to end
UNIT = "$"
PERIOD = "day"
PERIODS_PER_YEAR = 365

# ---- Search parameters (Bart §6.3.1) ----
D_MAX = 3                # max conditions in a description
Q = 20                   # result set size
W = 50                   # default beam width
U = 200                  # mini-clusters (pairs) for every BUS variant and both ensembles
POOL_SIZE = 30           # pool ensemble: conditions the BS step may use
W_MC = 5                 # per-mini-cluster ensemble: beam width of each small beam search

# ---- Narrow findings ----
MIN_COV = 0.02
MAX_COV = 0.15
MIN_STORES = 2
MIN_PEERS = 5

# ---- Reference point ----
START_DATE, END_DATE = "2013-01-01", "2016-05-22"   # base window = all of 2013
MIN_UNITS_PER_DAY = 0.5     # keep item x store series selling at least this many units per day in 2013
M5_DIR = "/kaggle/input/m5-forecasting-accuracy" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "M5"

# ---- Experiment 0: real data ----
MEASURES = ["M1", "M2", "M3"]
METHODS = ["BS", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]
NU = 20 if QUICK else 500                     # permutation runs for the DFD (Bart used 500)

# ---- Experiment 1: planted subgroups ----
PATTERNS = {"level": "M2", "trend": "M2", "timing": "M3"}    # pattern -> the measure that should catch it
SNR = [2, 8] if QUICK else [1, 2, 4, 8]                      # planted effect in units of the null spread sigma (see E1)
LITS = [2] if QUICK else [2, 3]                               # conditions in the planted description
OUTLIERS = [0.0] if QUICK else [0.0, 0.2]                     # share of planted records left unchanged
REPS = 1 if QUICK else 2                                      # different planted descriptions per setting
PLANT_COVERAGE = (0.02, 0.08)
E1_METHODS = ["BS", "BS w=10", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]

# ---- Experiment 2: interaction-only subgroups ----
E2_SNR = [8] if QUICK else [4, 8]
E2_REPS = 1 if QUICK else 3
E2_METHODS = ["BS w=5", "BS w=10", "BS w=20", "BS", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]

HIT_JACCARD = 0.8        # a planted subgroup counts as found if a subgroup with Jaccard >= 0.8 was evaluated
SEED = 0
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
RESULTS_DIR = "/kaggle/working/results_v3" if ON_KAGGLE else "results_v3"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
calendar = pd.read_csv(f"{M5_DIR}/calendar.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
sales_wide = pd.read_csv(f"{M5_DIR}/sales_train_evaluation.csv")
prices = pd.read_csv(f"{M5_DIR}/sell_prices.csv")

day_cols = [c for c in sales_wide.columns if c.startswith("d_")]
all_dates = calendar["date"].iloc[[int(c[2:]) - 1 for c in day_cols]].reset_index(drop=True)
in_period = (all_dates >= START_DATE) & (all_dates <= END_DATE)
dates = pd.DatetimeIndex(all_dates[in_period])
units = sales_wide[day_cols].to_numpy(dtype=np.float32)[:, in_period.to_numpy()]
BASE_LEN = int((dates < "2014-01-01").sum())

keep = units[:, :BASE_LEN].mean(axis=1) >= MIN_UNITS_PER_DAY
sales_wide, units = sales_wide[keep].reset_index(drop=True), units[keep]

# Dollars = units x the weekly shelf price of that item in that store (filled across weeks without a price).
week_of_day = calendar.set_index("date").loc[dates, "wm_yr_wk"].to_numpy()
weeks = np.unique(week_of_day)
price_wide = (prices.pivot_table(index=["store_id", "item_id"], columns="wm_yr_wk", values="sell_price")
                    .reindex(columns=weeks))
key = pd.MultiIndex.from_frame(sales_wide[["store_id", "item_id"]])
price_wide = price_wide.reindex(key).ffill(axis=1).bfill(axis=1)
price_daily = price_wide.to_numpy()[:, np.searchsorted(weeks, week_of_day)]
S = units.astype(float) * price_daily
MIN_BASE_LEVEL = 0.0       # the size filter was applied on units above

records = sales_wide[["store_id", "item_id", "dept_id", "cat_id", "state_id"]].rename(
    columns={"store_id": "store", "item_id": "item"})

# Item-level descriptors (same in every store): price level, price volatility, launch week.
item_prices = prices.groupby("item_id")["sell_price"]
first_week = prices.groupby("item_id")["wm_yr_wk"].min()
records["item_price"] = records["item"].map(item_prices.mean())
records["item_price_volatility"] = records["item"].map(item_prices.std() / item_prices.mean())
records["item_launch"] = records["item"].map(first_week.rank())

# Record-level descriptors (this store vs. the same item elsewhere): price level, price trend, discount frequency.
p = price_wide.to_numpy()
t_w = np.arange(p.shape[1], dtype=float)
t_w -= t_w.mean()
records["price_level"] = p.mean(axis=1)
records["price_trend"] = (p - p.mean(axis=1, keepdims=True)) @ t_w / (t_w ** 2).sum() * 52 / p.mean(axis=1)
records["discount_freq"] = (p < 0.95 * p.max(axis=1, keepdims=True)).mean(axis=1)

categorical_desc = [("state_id", "store"), ("cat_id", "item"), ("dept_id", "item")]
numeric_desc = [("item_price", "dept_id", "item"), ("item_price_volatility", None, "item"), ("item_launch", None, "item"),
                ("price_level", "item", "record"), ("price_trend", "item", "record"), ("discount_freq", "item", "record")]
del units, price_daily, sales_wide
print(S.shape, records.shape)

## Reference point: peer-expected sales
- **level_i** = normal sales per period in the base window (Konijn's cost *c*).
- **peer index** = how the *same item in the other stores* moved, relative to their own normal level (leave-one-out).
- **expected_i(t)** = level_i × peer index(t); **e_i(t) = actual − expected** (continuous, high-dimensional *c · t*);
  **r_i(t) = e_i(t) / level_i** (continuous *t*).
Seasonality and product mix are in the peer index, so they cancel; over the base window every record's excess sums to 0.

In [ ]:
level = S[:, :BASE_LEN].mean(axis=1)
item = records["item"].astype(str).to_numpy()

ok = level >= MIN_BASE_LEVEL
ok &= pd.Series(ok).groupby(item).transform("sum").to_numpy() >= MIN_PEERS
records = records[ok].reset_index(drop=True)
S, level, item = S[ok], level[ok], item[ok]
N, T = S.shape

group_sales = pd.DataFrame(S).groupby(item).transform("sum").to_numpy()
group_level = pd.Series(level).groupby(item).transform("sum").to_numpy()
peer_index = (group_sales - S) / (group_level - level)[:, None]
expected = level[:, None] * peer_index
excess = S - expected
rel_dev = np.clip(excess / level[:, None], -3, 3)
store_codes = pd.factorize(records["store"])[0]
del group_sales, peer_index, expected

print(f"{DATASET}: N = {N} records, T = {T} {PERIOD}s, {records['store'].nunique()} stores, "
      f"{records['item'].nunique()} items, base window = {BASE_LEN} {PERIOD}s")
print(f"Base-window excess per record (0 by construction): max |.| = {np.abs(excess[:, :BASE_LEN].sum(axis=1)).max():.6f}")

## Descriptors
Store and item are **row identifiers only**. Every descriptor has a *level* (store / item / record), used by the
permutation test. Numeric descriptors are z-discretized into five bins, either globally or *within a group*
(e.g. "price vs. the same item in other stores", "price vs. other items in the department").

In [ ]:
LABELS = ["very_low", "low", "medium", "high", "very_high"]


def z_bins(z):
    return pd.cut(z, bins=[-np.inf, -1.5, -0.5, 0.5, 1.5, np.inf], labels=LABELS).astype(str)


descriptor_cols, desc_level = [], {}
for column, level_ in categorical_desc:
    records[column] = records[column].astype(str)
    descriptor_cols.append(column)
    desc_level[column] = level_
for column, zgroup, level_ in numeric_desc:
    values = records[column].astype(float)
    if zgroup is None:
        z = (values - values.mean()) / values.std()
        name = f"{column}_cat"
    else:
        grouped = values.groupby(records[zgroup])
        z = (values - grouped.transform("mean")) / grouped.transform("std").replace(0, np.nan)
        name = f"{column}_vs_{zgroup}_cat"
    records[name] = z_bins(z.fillna(0))
    descriptor_cols.append(name)
    desc_level[name] = level_

A = records[descriptor_cols].astype(str).to_numpy(dtype=object)


def describe(desc):
    return " AND ".join(f"{descriptor_cols[j]} = {v}" for j, v in desc)


print(f"{len(descriptor_cols)} descriptors, {sum(records[c].nunique() for c in descriptor_cols)} conditions")
pd.DataFrame({"level": desc_level, "distinct values": records[descriptor_cols].nunique()})

## Quality measures and distances
- **M1** (Konijn MVPCD analogue): Σ e_i(t); **M2** (Relative CWTPD analogue): M1 minus what the complement did at the
  same scale, rewritten as L_T/(L_T − L_S) · Σ y_i; **M3** (Bart φ_cluster): α/(β+1) on the r_i curves.
- Exact optimistic estimates for M1/M2 (pruning); anti-monotone constraints (min coverage, min stores) are pruned,
  max coverage is only checked before reporting.
- Every quality object can **trace** its evaluations (time, Jaccard with a planted subgroup, valid?) for the
  anytime / equal-budget analysis.
- Distances are city-block, computed in parallel row blocks (float32 for large N).

In [ ]:
class QualityBase:
    def __init__(self, N, store_codes, min_cov=None, max_cov=None, min_stores=None):
        min_cov = MIN_COV if min_cov is None else min_cov
        max_cov = MAX_COV if max_cov is None else max_cov
        self.N = N
        self.store_codes = store_codes
        self.n_store_codes = int(store_codes.max()) + 1
        self.min_n = max(2, int(np.ceil(min_cov * N)))
        self.max_n = int(np.floor(max_cov * N))
        self.min_stores = MIN_STORES if min_stores is None else min_stores
        self.cache = {}
        self.track = None        # planted mask, if the evaluations should be traced
        self.trace = []          # (time, quality, jaccard with planted, valid)

    def n_stores(self, mask):
        return np.count_nonzero(np.bincount(self.store_codes[mask], minlength=self.n_store_codes))

    def expandable(self, mask):
        return mask.sum() >= self.min_n and self.n_stores(mask) >= self.min_stores

    def satisfies_constraints(self, mask):
        n = mask.sum()
        return n <= self.max_n and n < self.N and self.expandable(mask)

    def optimistic_estimate(self, mask):
        return None

    def __call__(self, mask):
        key = mask.tobytes()
        if key not in self.cache:
            self.cache[key] = self.compute(mask)
            if self.track is not None:
                inter = np.count_nonzero(mask & self.track)
                union = np.count_nonzero(mask | self.track)
                self.trace.append((time.time(), self.cache[key], inter / union, self.satisfies_constraints(mask)))
        return self.cache[key]


class MonetaryQuality(QualityBase):
    def __init__(self, excess, level, store_codes, relative=True, **constraints):
        super().__init__(len(level), store_codes, **constraints)
        self.relative = relative
        self.x = excess.sum(axis=1, dtype=np.float64)
        self.level = level
        self.L_T = level.sum()
        self.y = self.x - level * self.x.sum() / self.L_T

    def signed(self, mask):
        if not self.relative:
            return self.x[mask].sum()
        return self.y[mask].sum() * self.L_T / (self.L_T - self.level[mask].sum())

    def compute(self, mask):
        return abs(self.signed(mask))

    def optimistic_estimate(self, mask):
        v = self.y[mask] if self.relative else self.x[mask]
        bound = max(v[v > 0].sum(), -v[v < 0].sum())
        if self.relative:
            bound *= self.L_T / (self.L_T - self.level[mask].sum())
        return bound


class ClusterQuality(QualityBase):
    def __init__(self, D, store_codes, **constraints):
        super().__init__(len(D), store_codes, **constraints)
        self.D = D
        self.rowsum = D.sum(axis=1, dtype=np.float64)

    def components(self, mask):
        idx = np.flatnonzero(mask)
        n, n_c = len(idx), self.N - len(idx)
        within = self.D[np.ix_(idx, idx)].sum(dtype=np.float64)
        between = self.rowsum[idx].sum() - within
        return between / (n * n_c), within / (n * (n - 1))

    def compute(self, mask):
        alpha, beta = self.components(mask)
        return alpha / (beta + 1)


def _cityblock_block(Xb, X):
    return cdist(Xb, X, metric="cityblock")


def pairwise_cityblock(X, block=512):
    """Full N x N city-block distance matrix, row blocks computed in parallel."""
    N_ = len(X)
    dtype = np.float32 if N_ > 6000 else np.float64
    starts = list(range(0, N_, block))
    parts = Parallel(n_jobs=N_JOBS)(delayed(_cityblock_block)(X[s:s + block], X) for s in starts)
    D = np.empty((N_, N_), dtype=dtype)
    for s, part in zip(starts, parts):
        D[s:s + len(part)] = part
    return D


def update_rows(D, X, rows):
    """Distances after the target rows `rows` changed: only those rows / columns are recomputed."""
    D = D.copy()
    if len(rows):
        new = cdist(X[rows], X, metric="cityblock").astype(D.dtype)
        D[rows, :] = new
        D[:, rows] = new.T
    return D


start = time.time()
D_money = pairwise_cityblock(excess)
D_shape = pairwise_cityblock(rel_dev)
DISTANCE_TIME = time.time() - start
DISTANCE_FOR = {"M1": D_money, "M2": D_money, "M3": D_shape}
print(f"Distance matrices ({N} x {N}, T = {T}, {D_money.dtype}) built in {DISTANCE_TIME:.1f}s")


def make_quality(measure, exc=None, D=None):
    exc = excess if exc is None else exc
    if measure == "M1":
        return MonetaryQuality(exc, level, store_codes, relative=False)
    if measure == "M2":
        return MonetaryQuality(exc, level, store_codes, relative=True)
    return ClusterQuality(D_shape if D is None else D, store_codes)


_phi = make_quality("M2")
_mask = np.zeros(N, dtype=bool)
_mask[: N // 10] = True
_direct = _phi.x[_mask].sum() - level[_mask].sum() * (_phi.x.sum() - _phi.x[_mask].sum()) / (level.sum() - level[_mask].sum())
assert np.isclose(_phi.signed(_mask), _direct)

## Search strategies
| method | candidates come from |
|---|---|
| **BS**, **BS w=k** | beam search (Alg. 1) with beam width 50, or k |
| **BUS** | the u closest pairs, every combination of their shared conditions (Alg. 2/3) |
| **BUS-α** | as BUS with α-matrix pairs: similar *and* unusual (§4.3.2) |
| **BUS→BS pool** | BUS-α pairs vote for conditions → top 30 → one beam search over them |
| **BUS→BS per-MC** | a beam search (width 5) per BUS-α mini-cluster over its shared conditions (§4.3.3) |

Pairs are selected block-wise (no N² index arrays), so this scales to large N. Pruning as in v2:
coverage (Apriori), optimistic estimate (M1/M2), redundancy.

In [ ]:
def mask_of(A, desc):
    mask = np.ones(len(A), dtype=bool)
    for j, v in desc:
        mask &= A[:, j] == v
    return mask


class ResultSet:
    def __init__(self, q):
        self.q = q
        self.items = {}

    def insert(self, quality, desc, mask):
        key = mask.tobytes()
        old = self.items.get(key)
        if old is not None and len(old[1]) <= len(desc):
            return
        self.items[key] = (quality, desc, mask)
        if len(self.items) > self.q:
            del self.items[min(self.items, key=lambda k: self.items[k][0])]

    def full(self):
        return len(self.items) >= self.q

    def threshold(self):
        return min(v[0] for v in self.items.values()) if self.full() else -np.inf

    def sorted(self):
        return sorted(self.items.values(), key=lambda x: -x[0])


def oe_below_threshold(phi, mask, result_set):
    if not result_set.full():
        return False
    oe = phi.optimistic_estimate(mask)
    return oe is not None and oe < result_set.threshold()


def refine(seed, seed_mask, A, pool=None):
    used = {j for j, _ in seed}
    if pool is None:
        options = ((j, v) for j in range(A.shape[1]) if j not in used for v in pd.unique(A[seed_mask, j]))
    else:
        options = ((j, v) for j, v in pool if j not in used and (A[seed_mask, j] == v).any())
    for condition in options:
        yield tuple(sorted(seed + (condition,), key=lambda c: c[0]))


def _beam(A, phi, pool, w, d, prune, result_set, stats, evaluated_keys, mask_cache):
    def get_mask(desc):
        if desc not in mask_cache:
            mask_cache[desc] = mask_of(A, desc)
        return mask_cache[desc]

    candidate_queue = [()]
    seen, tie = set(), itertools.count()
    for level_ in range(1, d + 1):
        beam = []
        for seed in candidate_queue:
            seed_mask = get_mask(seed)
            if prune and seed and oe_below_threshold(phi, seed_mask, result_set):
                stats["pruned_oe_seed"] += 1
                continue
            for desc in refine(seed, seed_mask, A, pool):
                if desc in seen:
                    continue
                seen.add(desc)
                stats["generated"] += 1
                mask = get_mask(desc)
                if not phi.expandable(mask):
                    stats["skipped_coverage"] += 1
                    continue
                if prune and seed and mask.sum() == seed_mask.sum():
                    stats["pruned_redundant"] += 1
                    continue
                quality = phi(mask)
                stats["visits"] += 1
                evaluated_keys.add(desc)
                if phi.satisfies_constraints(mask):
                    result_set.insert(quality, desc, mask)
                if prune and oe_below_threshold(phi, mask, result_set):
                    stats["pruned_oe"] += 1
                    continue
                item_ = (quality, next(tie), desc)
                if len(beam) < w:
                    heapq.heappush(beam, item_)
                else:
                    heapq.heappushpop(beam, item_)
        candidate_queue = [desc for _, _, desc in sorted(beam, reverse=True)]


def beam_search(A, phi, w=W, d=D_MAX, q=Q, prune=False, pool=None):
    stats, result_set, keys = Counter(), ResultSet(q), set()
    _beam(A, phi, pool, w, d, prune, result_set, stats, keys, {})
    stats["evaluated"] = len(keys)
    return result_set.sorted(), stats


def top_pairs(D, u=U, kind="alpha", block=512):
    """Block-wise pair selection over the lower triangle.
    kind="closest": the u smallest distances (plain BUS, Alg. 2).
    kind="alpha":   the u largest alpha_ij / (D_ij + 1) (alpha-matrix, §4.3.2)."""
    N_ = len(D)
    rowsum = D.sum(axis=1, dtype=np.float64)
    scores, rows_, cols_ = [], [], []
    for s in range(0, N_, block):
        Db = D[s:s + block].astype(np.float64)
        i = np.arange(s, s + len(Db))[:, None]
        j = np.arange(N_)[None, :]
        if kind == "closest":
            score = -Db
        else:
            alpha = (rowsum[i] + rowsum[j] - 2 * Db) / (2 * (N_ - 2))
            score = alpha / (Db + 1)
        score = np.where(j < i, score, -np.inf).ravel()
        k = min(u, int(np.isfinite(score).sum()))
        if k == 0:
            continue
        best = np.argpartition(-score, k - 1)[:k]
        scores.append(score[best])
        rows_.append(s + best // N_)
        cols_.append(best % N_)
    scores, rows_, cols_ = np.concatenate(scores), np.concatenate(rows_), np.concatenate(cols_)
    order = np.argsort(-scores)[:u]
    return [(int(rows_[o]), int(cols_[o])) for o in order]


def shared_conditions(mc, A):
    first = mc[0]
    return [(j, A[first, j]) for j in range(A.shape[1]) if all(A[i, j] == A[first, j] for i in mc)]


def evaluate_candidates(A, phi, candidates, q=Q, prune=False):
    stats = Counter(generated=len(candidates))
    result_set = ResultSet(q)
    dead = set()
    for desc in sorted(candidates, key=len):
        if prune and len(desc) > 1 and any(
            sub in dead for size in range(1, len(desc)) for sub in itertools.combinations(desc, size)
        ):
            stats["pruned_superset"] += 1
            continue
        mask = mask_of(A, desc)
        if not phi.expandable(mask):
            stats["skipped_coverage"] += 1
            dead.add(desc)
            continue
        quality = phi(mask)
        stats["evaluated"] += 1
        stats["visits"] += 1
        if phi.satisfies_constraints(mask):
            result_set.insert(quality, desc, mask)
        if prune and oe_below_threshold(phi, mask, result_set):
            dead.add(desc)
    return result_set.sorted(), stats


def bus(A, phi, pairs, d=D_MAX, q=Q, prune=False):
    candidates = set()
    for mc in pairs:
        conditions = shared_conditions(mc, A)
        for size in range(1, d + 1):
            candidates.update(itertools.combinations(conditions, size))
    top, stats = evaluate_candidates(A, phi, candidates, q, prune)
    stats["mini_clusters"] = len(pairs)
    return top, stats


def ensemble_pool(A, phi, pairs, pool_size=POOL_SIZE, prune=False):
    votes = Counter(c for mc in pairs for c in shared_conditions(mc, A))
    pool = [c for c, _ in votes.most_common(pool_size)]
    top, stats = beam_search(A, phi, prune=prune, pool=pool)
    stats["mini_clusters"] = len(pairs)
    stats["pool"] = len(pool)
    return top, stats, pool


def ensemble_per_mc(A, phi, pairs, w=W_MC, d=D_MAX, q=Q, prune=False):
    stats, result_set, keys, mask_cache = Counter(), ResultSet(q), set(), {}
    spaces = set()
    for mc in pairs:
        space = tuple(shared_conditions(mc, A))
        if not space:
            stats["mc_empty"] += 1
            continue
        if space in spaces:
            stats["mc_duplicate_space"] += 1
            continue
        spaces.add(space)
        _beam(A, phi, list(space), w, d, prune, result_set, stats, keys, mask_cache)
    stats["evaluated"] = len(keys)
    stats["mini_clusters"] = len(pairs)
    stats["mc_spaces"] = len(spaces)
    union = sorted({c for space in spaces for c in space}, key=lambda c: (c[0], str(c[1])))
    return result_set.sorted(), stats, union


def run_method(method, A, phi, D, prune, pairs=None):
    """method: "BS", "BS w=<k>", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC".
    Runtime includes mini-cluster selection unless precomputed pairs are passed."""
    start = time.time()
    pool = None
    if method.startswith("BS"):
        width = int(method.split("w=")[1]) if "w=" in method else W
        top, stats = beam_search(A, phi, w=width, prune=prune)
    else:
        if pairs is None:
            pairs = top_pairs(D, kind="closest" if method == "BUS" else "alpha")
        stats_pair = time.time() - start
        if method in ("BUS", "BUS-α"):
            top, stats = bus(A, phi, pairs, prune=prune)
        elif method == "BUS→BS pool":
            top, stats, pool = ensemble_pool(A, phi, pairs, prune=prune)
        else:
            top, stats, pool = ensemble_per_mc(A, phi, pairs, prune=prune)
        stats["pair_time"] = stats_pair
    stats["runtime"] = time.time() - start
    return top, stats, pool


def top_keys(top):
    return {m.tobytes() for _, _, m in top}


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 1.0

In [ ]:
_X = excess.sum(axis=1, dtype=np.float64)
_X_T, _L_T = _X.sum(), level.sum()
_shape_rowsum = D_shape.sum(axis=1, dtype=np.float64)
_shape_overall = _shape_rowsum.sum() / (N * (N - 1))
_m2 = make_quality("M2")


def subgroup_metrics(top, measure, method, prune, p_values=None):
    """Per subgroup: Konijn money metrics, Bart's alpha / beta / phi (App. E), coverage, complement, lift, p-value."""
    rows = []
    for rank, (quality, desc, mask) in enumerate(top, start=1):
        idx = np.flatnonzero(mask)
        n, n_c = len(idx), N - len(idx)
        within = D_shape[np.ix_(idx, idx)].sum(dtype=np.float64)
        alpha = (_shape_rowsum[idx].sum() - within) / (n * n_c)
        beta = within / (n * (n - 1))
        X_s, L_s = _X[mask].sum(), level[mask].sum()
        rows.append({
            "dataset": DATASET, "measure": measure, "method": method, "prune": prune, "rank": rank,
            "description": describe(desc), "length": len(desc), "quality": quality,
            "excess_raw": X_s, "excess_relative": _m2.signed(mask),
            "pct_of_normal": 100 * X_s / (L_s * T),
            "complement_pct_of_normal": 100 * (_X_T - X_s) / ((_L_T - L_s) * T),
            "coverage_n": n, "coverage": n / N, "n_stores": _m2.n_stores(mask), "complement_n": n_c,
            "alpha": alpha, "beta": beta, "phi_cluster": alpha / (beta + 1), "lift": _shape_overall / beta,
            "p_value": p_values[rank - 1] if p_values is not None else np.nan,
        })
    return pd.DataFrame(rows)

## Experiment 0: real data — every measure × search × pruning off/on (Bart Table 6.9 metrics + steps)

In [ ]:
main_rows, main_runs, metric_tables = [], {}, []
for measure in MEASURES:
    D = DISTANCE_FOR[measure]
    for method in METHODS:
        for prune in (False, True):
            phi = make_quality(measure)
            top, stats, pool = run_method(method, A, phi, D, prune)
            main_runs[(measure, method, prune)] = (top, stats, pool, phi)

    reference = top_keys(main_runs[(measure, "BS", False)][0])
    bs_conditions = {c for _, desc, _ in main_runs[(measure, "BS", False)][0] for c in desc}
    for method in METHODS:
        for prune in (False, True):
            top, stats, pool, phi = main_runs[(measure, method, prune)]
            qualities = np.array([q for q, _, _ in top]) if top else np.array([np.nan])
            coverages = np.array([m.mean() for _, _, m in top]) if top else np.array([np.nan])
            main_rows.append({
                "dataset": DATASET, "N": N, "T": T, "distance_time_s": DISTANCE_TIME,
                "measure": measure, "method": method, "prune": prune,
                "runtime_s": stats["runtime"], "pair_time_s": stats.get("pair_time", 0.0),
                "generated": stats.get("generated", 0), "evaluated": stats["evaluated"],
                "visits": stats.get("visits", 0),
                "pruned": sum(v for k, v in stats.items() if k.startswith("pruned")),
                "mini_clusters": stats.get("mini_clusters", 0), "mc_spaces": stats.get("mc_spaces", np.nan),
                "max_quality": qualities.max(), "avg_quality": qualities.mean(),
                "max_coverage": coverages.max(), "avg_coverage": coverages.mean(), "n_results": len(top),
                "jaccard_vs_BS": jaccard(top_keys(top), reference), "shared_vs_BS": len(top_keys(top) & reference),
                "pool_recall": (len(bs_conditions & set(pool)) / len(bs_conditions)) if pool else np.nan,
            })
            if prune:
                metric_tables.append(subgroup_metrics(top, measure, method, prune))

main_df = pd.DataFrame(main_rows)
main_df.to_csv(f"{RESULTS_DIR}/{DATASET}_main.csv", index=False)
display(main_df.drop(columns=["dataset", "N", "T", "distance_time_s"]).round(3))

## Experiment 0b: significance with Bart's 500-run DFD (store / item / record-level permutation)
Each descriptor is permuted at its own level: store-level descriptors are handed to a random other store,
item-level ones to a random other item (consistently across stores), record-level ones are shuffled among
the records of the same item. The target, the quality object and the mini-cluster pairs stay fixed.
M2, pruning on; the permutation runs are spread over all CPU cores. p-values follow Bart eq. 4.6.

In [ ]:
def permuted_descriptors(rng):
    perm = records[descriptor_cols].copy()
    for key, lvl in (("store", "store"), ("item", "item")):
        cols = [c for c in descriptor_cols if desc_level[c] == lvl]
        if cols:
            table = records.groupby(key)[cols].first()
            donor = dict(zip(table.index, rng.permutation(table.index)))
            perm[cols] = table.loc[records[key].map(donor)].to_numpy()
    for c in [c for c in descriptor_cols if desc_level[c] == "record"]:
        perm[c] = records.groupby("item")[c].transform(lambda s: rng.permutation(s.to_numpy()))
    return perm.astype(str).to_numpy(dtype=object)


def _dfd_chunk(method, A_list, phi, pairs):
    return [(run_method(method, A_perm, phi, None, True, pairs=pairs)[0] or [(0.0,)])[0][0] for A_perm in A_list]


def empirical_p_value(quality, dfd):
    return ((dfd > quality).sum() + 0.5 * (dfd == quality).sum() + 1) / (len(dfd) + 1)


rng = np.random.default_rng(SEED)
permuted = [permuted_descriptors(rng) for _ in range(NU)]      # the same 500 nulls for every method
chunks = [permuted[i::N_JOBS] for i in range(N_JOBS)]
pairs_m2 = {"BUS": top_pairs(D_money, kind="closest"), "alpha": top_pairs(D_money, kind="alpha")}

dfd_rows, dfd_values = [], {}
for method in METHODS:
    top, stats, pool, phi = main_runs[("M2", method, True)]
    pairs = None if method == "BS" else (pairs_m2["BUS"] if method == "BUS" else pairs_m2["alpha"])
    start = time.time()
    parts = Parallel(n_jobs=N_JOBS)(delayed(_dfd_chunk)(method, chunk, make_quality("M2"), pairs) for chunk in chunks)
    dfd = np.array([v for part in parts for v in part])
    dfd_values[method] = dfd
    p_values = [empirical_p_value(q, dfd) for q, _, _ in top]
    metric_tables = [t for t in metric_tables if not (len(t) and t["measure"].iloc[0] == "M2" and t["method"].iloc[0] == method)]
    metric_tables.append(subgroup_metrics(top, "M2", method, True, p_values))
    dfd_rows.append({"dataset": DATASET, "method": method, "nu": NU, "dfd_seconds": time.time() - start,
                     "dfd_median": np.median(dfd), "dfd_95": np.quantile(dfd, 0.95), "dfd_99": np.quantile(dfd, 0.99),
                     "best_quality": top[0][0] if top else np.nan,
                     "n_significant_005": int(np.sum(np.array(p_values) < 0.05)),
                     "n_significant_001": int(np.sum(np.array(p_values) < 0.01))})
    print(f"{method}: {NU} permutations in {time.time() - start:.0f}s, "
          f"{dfd_rows[-1]['n_significant_005']} of {len(top)} with p < 0.05")

dfd_df = pd.DataFrame(dfd_rows)
dfd_df.to_csv(f"{RESULTS_DIR}/{DATASET}_dfd.csv", index=False)
pd.DataFrame(dfd_values).to_csv(f"{RESULTS_DIR}/{DATASET}_dfd_values.csv", index=False)
metrics_df = pd.concat(metric_tables, ignore_index=True)
metrics_df.to_csv(f"{RESULTS_DIR}/{DATASET}_subgroups.csv", index=False)
display(dfd_df.round(3))
display(metrics_df[(metrics_df["measure"] == "M2") & (metrics_df["method"] == "BS")].head(10).round(3))

## Experiments 1 and 2: planted subgroups in a null background (semi-synthetic)
As in Bart's synthetic study (§5.2, §6.2), one **true subgroup** is planted and the question is whether a method
ranks it. To keep the real noise, seasonality and scale, the background is the real target with the descriptors
**permuted** (one draw of the permutation null above), so no other subgroup is truly exceptional. Then a known
deviation is added to the records of a random description, as a share of each record's normal sales.
Effect sizes are **calibrated**: δ = snr × σ, where σ is how much random described subgroups already vary in the
null background (σ_rate for level/trend, σ_timing for timing), with snr ∈ {1, 2, 4, 8}:
- **level**: a step after the base window (+δ); **trend**: a ramp to +2δ (average δ) — money patterns → **M2**;
- **timing**: a yearly wave ±δ that sums to ~0, so only the *shape* changes → **M3** (money measures cannot see it);
- **outliers**: a share of the described records is left unchanged.

**Experiment 2 (interaction only):** +δ in A∧B, and compensating negative deviations in A∧¬B and ¬A∧B so that the
planted money over A and over B is exactly zero. A and B alone look unremarkable, so a greedy beam can drop them at
level 1; only the combination is exceptional. Beam searches with widths 5, 10, 20, 50 are compared with BUS and both ensembles.

**Recorded per run** (a subgroup "is" the planted one if its records have Jaccard ≥ 0.8 with the planted records):
- `reported` / `rank`: in the final top-20 and at which rank (Bart's "rank of the true subgroup");
- `rank_at_<B>`: its rank among everything evaluated within the first B quality evaluations — what an
  equal-budget run stopped at B would report (Experiment 3); `rank_at_<t>s`: the same within t seconds;
- `evaluated`, `runtime_s`. Planted datasets run in parallel; each job runs all methods one after another.

In [ ]:
POST = np.arange(T) >= BASE_LEN
BUDGETS = [25, 50, 100, 200, 400, 800, 1600, 3200, 6400]
SECONDS = [0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 30, 60]


def pattern_curve(kind, delta):
    t = np.arange(T)
    if kind == "level":
        return delta * POST
    if kind == "trend":
        return 2 * delta * np.clip((t - BASE_LEN + 1) / (T - BASE_LEN), 0, None) * POST
    return delta * np.sin(2 * np.pi * t / PERIODS_PER_YEAR)


def n_stores_of(mask):
    return np.count_nonzero(np.bincount(store_codes[mask], minlength=int(store_codes.max()) + 1))


def sample_description(rng, lits, A_, coverage=PLANT_COVERAGE, tries=20000):
    # A random description of `lits` conditions taken from one record, with coverage in range,
    # at least MIN_STORES stores, and every condition actually narrowing the subgroup.
    for _ in range(tries):
        r = int(rng.integers(N))
        cols = rng.choice(A_.shape[1], lits, replace=False)
        desc = tuple(sorted(((int(j), A_[r, j]) for j in cols), key=lambda c: c[0]))
        mask = mask_of(A_, desc)
        if not (coverage[0] <= mask.mean() <= coverage[1]) or n_stores_of(mask) < MIN_STORES:
            continue
        if any(mask_of(A_, tuple(c for c in desc if c != drop)).sum() == mask.sum() for drop in desc):
            continue
        return desc, mask
    return None, None


def sample_interaction(rng, A_, coverage=PLANT_COVERAGE, tries=20000):
    # Conditions A, B on different attributes: A∧B in the coverage range, A∧¬B and ¬A∧B larger than A∧B.
    for _ in range(tries):
        r = int(rng.integers(N))
        j1, j2 = rng.choice(A_.shape[1], 2, replace=False)
        a, b = (int(j1), A_[r, j1]), (int(j2), A_[r, j2])
        mA, mB = mask_of(A_, (a,)), mask_of(A_, (b,))
        mAB = mA & mB
        if not (coverage[0] <= mAB.mean() <= coverage[1]) or n_stores_of(mAB) < MIN_STORES:
            continue
        if (mA & ~mB).sum() < mAB.sum() or (~mA & mB).sum() < mAB.sum():
            continue
        return (a, b), mA, mB, mAB
    return None, None, None, None


def _rank_of_planted(entries):
    # entries: (time, quality, jaccard, valid). Rank of the best planted match among the valid evaluated
    # subgroups (1 = best); NaN if the planted subgroup was not evaluated.
    planted_q = [q for _, q, jac, valid in entries if valid and jac >= HIT_JACCARD]
    if not planted_q:
        return np.nan
    best = max(planted_q)
    return 1 + sum(1 for _, q, jac, valid in entries if valid and jac < HIT_JACCARD and q > best)


def _plant_job(spec, excess, level, D_money, D_shape, store_codes, methods):
    # Builds one planted dataset (null background + planted deviation) and runs every method (pruning on).
    rng = np.random.default_rng(spec["seed"])
    A_ = spec["A"]
    exc = excess.copy()
    changed = []
    for rows, scale in spec["plants"]:
        rows = np.asarray(rows)
        keep = rows[rng.random(len(rows)) >= spec["outliers"]]
        exc[keep] += scale * spec["curve"][None, :] * level[keep, None]
        changed.append(keep)
    changed = np.unique(np.concatenate(changed))
    if spec["measure"] == "M2":
        D = update_rows(D_money, exc, changed)
    else:
        D = update_rows(D_shape, np.clip(exc / level[:, None], -3, 3), changed)
    planted = np.zeros(len(level), dtype=bool)
    planted[spec["target_rows"]] = True

    out = []
    for method in methods:
        if spec["measure"] == "M2":
            phi = MonetaryQuality(exc, level, store_codes, relative=True)
        else:
            phi = ClusterQuality(D, store_codes)
        phi.track, phi.trace = planted, []
        t0 = time.time()
        top, stats, pool = run_method(method, A_, phi, D, True)
        trace = phi.trace
        top_jac = [np.count_nonzero(m & planted) / np.count_nonzero(m | planted) for _, _, m in top]
        reported_ranks = [r + 1 for r, jac in enumerate(top_jac) if jac >= HIT_JACCARD]
        row = {**{k: v for k, v in spec.items() if k not in ("plants", "curve", "target_rows", "A")},
               "method": method, "planted_n": int(planted.sum()),
               "evaluated_hit": any(valid and jac >= HIT_JACCARD for _, _, jac, valid in trace),
               "reported": bool(reported_ranks), "rank": reported_ranks[0] if reported_ranks else np.nan,
               "rank_all": _rank_of_planted(trace), "best_jaccard_top": max(top_jac, default=0.0),
               "evaluated": len(trace), "runtime_s": stats["runtime"], "best_quality": top[0][0] if top else np.nan}
        for b in BUDGETS:
            row[f"rank_at_{b}"] = _rank_of_planted(trace[:b])
        for s in SECONDS:
            row[f"rank_at_{s}s"] = _rank_of_planted([e for e in trace if e[0] - t0 <= s])
        out.append(row)
    return out


def run_plant_specs(specs, methods):
    jobs = max(1, min(N_JOBS, len(specs)))
    parts = Parallel(n_jobs=jobs)(
        delayed(_plant_job)(spec, excess, level, D_money, D_shape, store_codes, methods) for spec in specs)
    return pd.DataFrame([row for part in parts for row in part])


# ---------------- Calibration: how much do random described subgroups vary in the null background?
# sigma_rate:   spread of the post-base excess rate (excess / normal sales) of random subgroups  -> level / trend
# sigma_timing: spread of the yearly-wave amplitude of their mean relative deviation             -> timing
# A planted effect of snr x sigma is then equally "subtle" on every dataset.
rng = np.random.default_rng(SEED + 1)
_sin = np.sin(2 * np.pi * np.arange(T) / PERIODS_PER_YEAR)
rates, amps = [], []
for _ in range(20):
    A_null = permuted_descriptors(rng)
    for _ in range(10):
        desc, mask = sample_description(rng, int(rng.choice(LITS)), A_null)
        if desc is not None:
            rates.append(excess[mask][:, POST].sum() / (level[mask].sum() * POST.sum()))
            amps.append(2 * (rel_dev[mask].mean(axis=0) * _sin).mean())
SIGMA = {"rate": float(np.std(rates)), "timing": float(np.std(amps))}
print(f"null spread: sigma_rate = {SIGMA['rate']:.4f} ({100 * SIGMA['rate']:.2f}% of normal sales), "
      f"sigma_timing = {SIGMA['timing']:.4f}")

# ---------------- Experiment 1: one planted description per dataset
e1_specs = []
for pattern, measure in PATTERNS.items():
    for snr in SNR:
        delta = snr * SIGMA["timing" if pattern == "timing" else "rate"]
        for lits in LITS:
            for outliers in OUTLIERS:
                for rep in range(REPS):
                    A_null = permuted_descriptors(rng)
                    desc, mask = sample_description(rng, lits, A_null)
                    if desc is None:
                        continue
                    rows = np.flatnonzero(mask)
                    e1_specs.append({"experiment": "E1", "dataset": DATASET, "pattern": pattern, "measure": measure,
                                     "snr": snr, "delta": delta, "lits": lits, "outliers": outliers, "rep": rep,
                                     "description": describe(desc), "seed": int(rng.integers(1 << 31)), "A": A_null,
                                     "curve": pattern_curve(pattern, delta), "plants": [(rows, 1.0)], "target_rows": rows})
start = time.time()
e1_df = run_plant_specs(e1_specs, E1_METHODS)
e1_df.to_csv(f"{RESULTS_DIR}/{DATASET}_e1.csv", index=False)
print(f"E1: {len(e1_specs)} planted datasets x {len(E1_METHODS)} methods in {time.time() - start:.0f}s")

# ---------------- Experiment 2: interaction-only planted subgroup
e2_specs = []
for snr in E2_SNR:
    delta = snr * SIGMA["rate"]
    for rep in range(E2_REPS):
        A_null = permuted_descriptors(rng)
        conds, mA, mB, mAB = sample_interaction(rng, A_null)
        if conds is None:
            continue
        L_AB, L_AnB, L_nAB = level[mAB].sum(), level[mA & ~mB].sum(), level[~mA & mB].sum()
        e2_specs.append({"experiment": "E2", "dataset": DATASET, "pattern": "interaction", "measure": "M2",
                         "snr": snr, "delta": delta, "lits": 2, "outliers": 0.0, "rep": rep,
                         "description": describe(conds), "seed": int(rng.integers(1 << 31)), "A": A_null,
                         "curve": pattern_curve("level", delta),
                         "plants": [(np.flatnonzero(mAB), 1.0),
                                    (np.flatnonzero(mA & ~mB), -L_AB / L_AnB),
                                    (np.flatnonzero(~mA & mB), -L_AB / L_nAB)],
                         "target_rows": np.flatnonzero(mAB)})
start = time.time()
e2_df = run_plant_specs(e2_specs, E2_METHODS)
e2_df.to_csv(f"{RESULTS_DIR}/{DATASET}_e2.csv", index=False)
print(f"E2: {len(e2_specs)} planted datasets x {len(E2_METHODS)} methods in {time.time() - start:.0f}s")

## Results of Experiments 1–3
- **E1/E2:** share of planted subgroups reported in the top-20 (`found`), share ranked first (`top1`), mean rank,
  evaluations and runtime per method.
- **E3 (equal budget / anytime):** share of planted subgroups that would be in the top-20 if the run were stopped
  after B quality evaluations, or after t seconds (mini-cluster selection included).

In [ ]:
def summarise(df):
    return (df.assign(found=df["rank"] <= Q, top1=df["rank"] == 1)
              .groupby(["pattern", "snr", "method"], sort=False)
              .agg(found=("found", "mean"), top1=("top1", "mean"), mean_rank=("rank", "mean"),
                   evaluated=("evaluated", "mean"), runtime_s=("runtime_s", "mean"), runs=("found", "size"))
              .round(3))


print("Experiment 1 (single planted description)")
display(summarise(e1_df))
print("Experiment 2 (interaction only)")
display(summarise(e2_df))

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for row, (name, df) in enumerate([("E1", e1_df), ("E2", e2_df)]):
    for col, (grid, suffix, label) in enumerate([(BUDGETS, "", "quality evaluations"), (SECONDS, "s", "seconds")]):
        for method, g in df.groupby("method", sort=False):
            values = [np.mean(g[f"rank_at_{b}{suffix}"] <= Q) for b in grid]
            axes[row, col].plot(grid, values, marker="o", label=method)
        axes[row, col].set_xscale("log")
        axes[row, col].set_ylim(0, 1.02)
        axes[row, col].set_xlabel(f"budget ({label})")
        axes[row, col].set_ylabel("share of planted subgroups in the top-20")
        axes[row, col].set_title(f"{DATASET} | {name}: found within budget")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()